# Combinatorial Alpha Pipeline v2

**llm_alpha_guide.ipynb 구조를 차용한 조합 알파 생성 파이프라인**

## 파이프라인 흐름

```
10번의 Dataset_combine Batch
    |
    v
[1. Dataset 선택] 2개 이상이면 랜덤 2개 선택 (미만이면 스킵)
    |
    v
[2. Seed 추출] 각 dataset에서 0-fail 알파 랜덤 10개 (총 20개)
    |
    v
[3. LLM 생성] 20개 seed -> 100개 조합 생성 (2~4개 seed 자유 조합)
    |
    v
[4. Sanity Check] Type validation
    |
    v
[5. Simulation] 10개씩 배치 시뮬레이션 (1-10, 11-20, ..., 91-100)
    |
    v
[6. Save] PASS(0-fail)만 누적 저장 (warning 포함)
```

## 특징

- **Resume 지원**: `gen_json/combinatorial_batch_{idx}.json`
- **PowerPool 회피**: rank, group_zscore, min, max 사용 권장
- **operators_list.json 강제**: 허용된 operator만 사용

## 1. Imports

In [119]:
import json
import os
import re
import sys
import random
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))

import ace_lib as ace
import llm_functions as llm
from parser import tree_node

print("Imports complete")

Imports complete


## 2. Configuration

In [120]:
# ============================================================
# Configuration
# ============================================================

# Path settings
SCRIPT_DIR = Path('.').resolve()
OUTPUT_DIR = SCRIPT_DIR / "results" / "combinatorial"
GEN_JSON_DIR = SCRIPT_DIR / "gen_json"

# Dataset files (hardcoded)
DATASET_FILES = {
    'model25': SCRIPT_DIR / 'model25.txt',
    'model30': SCRIPT_DIR / 'model30.txt',
    'model138': SCRIPT_DIR / 'model138.txt',
    'analyst39': SCRIPT_DIR / 'analyst39.txt',
}

# Resource files
OPERATORS_FILE = SCRIPT_DIR / 'operators_list.json'
REGION = "EUR"
UNIVERSE = "TOP2500"
DELAY = 1
DATAFIELDS_FILE = SCRIPT_DIR / f'datafield/{DELAY}/{REGION}/{UNIVERSE}/{DELAY}_{REGION}_{UNIVERSE}_total.json'

# Pipeline parameters
NUM_BATCHES = 100                  # Total number of Dataset_combine batches
SEEDS_PER_DATASET = 10              # Seeds to extract from each dataset
COMBINATIONS_PER_BATCH = 100        # Combinations LLM generates per batch
SIMULATION_BATCH_SIZE = 8           # Simulate 8 at a time (matches llm_alpha_guide)
RANDOM_SEED = 42                    # Fixed seed for reproducibility
CONCURRENCY = 8                     # Brain API concurrent simulations (matches llm_alpha_guide)

# LLM settings
GPT_MODEL = 'gpt-4o'

# Output file (cumulative)
OUTPUT_TXT_FILE = OUTPUT_DIR / "combined_alphas.txt"

# ============================================================
# Validate files
# ============================================================
print("="*70)
print("CONFIGURATION")
print("="*70)

# Filter valid datasets
valid_datasets = {}
for name, path in DATASET_FILES.items():
    if path.exists():
        valid_datasets[name] = path
        print(f"  [OK] {name}: {path.name}")
    else:
        print(f"  [MISSING] {name}: {path.name}")

DATASET_FILES = valid_datasets
print(f"\nValid datasets: {len(DATASET_FILES)}")

if len(DATASET_FILES) < 2:
    print("\n[ERROR] Need at least 2 datasets!")

print(f"\nPipeline:")
print(f"  Batches: {NUM_BATCHES}")
print(f"  Seeds per dataset: {SEEDS_PER_DATASET}")
print(f"  Combinations per batch: {COMBINATIONS_PER_BATCH}")
print(f"  Simulation batch size: {SIMULATION_BATCH_SIZE}")
print(f"  GPT Model: {GPT_MODEL}")
print(f"  Random seed: {RANDOM_SEED}")
print("="*70)

CONFIGURATION
  [OK] model25: model25.txt
  [OK] model30: model30.txt
  [OK] model138: model138.txt
  [OK] analyst39: analyst39.txt

Valid datasets: 4

Pipeline:
  Batches: 100
  Seeds per dataset: 10
  Combinations per batch: 100
  Simulation batch size: 8
  GPT Model: gpt-4o
  Random seed: 42


## 3. Constants & Patterns

In [121]:
# Regex to parse 0-fail alphas from .txt files
ZERO_FAIL_BLOCK_PATTERN = re.compile(
    r"--- #(\d+) \| FAIL: 0[^-]*---\n"
    r"ID: ([A-Za-z0-9]+)\n"
    r"Region: ([A-Z]+), Universe: ([A-Z0-9]+)\n"
    r"Sharpe: ([\d.-]+), Fitness: ([\d.-]+), Turnover: ([\d.-]+)\n"
    r"Expression: (.+?)\n",
    re.DOTALL
)

print("Constants defined")

Constants defined


## 4. Sanity Check Functions

In [122]:
def return_type(node, operators, datafields):
    """Determine the return type of a tree node"""
    if node.node_type == 'operator':
        return operators[node.value]['output']
    if node.node_type == 'datafield':
        return datafields[node.value]['type']
    if node.node_type in ['number', 'string']:
        return "NUMBER"
    if node.node_type == 'special_argument':
        return "SPECIAL_ARGUMENT"


def check_input(operator_inputs, children_types, _debug=False):
    """Check if operator inputs match children types"""
    while len(operator_inputs) != 0:
        if operator_inputs[0] == []:
            operator_inputs.pop(0)
            children_types.pop(0)
        elif children_types[0] in operator_inputs[0]:
            operator_inputs.pop(0)
            children_types.pop(0)
        else:
            return False
    return True


def sanity_checker(exp: str, operators: dict, datafields: dict) -> Tuple[bool, Optional[str]]:
    """
    Validate expression using tree parsing and type checking
    Returns: (is_valid, error_message)
    """
    try:
        exp_tree = tree_node(exp)

        for node in [n for n in exp_tree.collect_all_nodes() if n.node_type == "operator"]:
            operator_name = node.value

            if operator_name not in operators:
                return False, f"Unknown operator: {operator_name}"

            operator_input_types = eval(operators[operator_name]['input'])
            children_return_types = [return_type(x, operators, datafields) for x in node.children]

            if not check_input(operator_input_types, children_return_types):
                return False, f"Type mismatch in operator: {operator_name}"

        final_type = return_type(exp_tree, operators, datafields)
        if final_type != "MATRIX":
            return False, f"Final output type is {final_type}, not MATRIX"

        return True, None

    except Exception as e:
        return False, f"Parsing error: {str(e)}"


def extract_sim_record(sim_result):
    """
    Extract simulation record from Brain API result (matching llm_alpha_guide.ipynb).
    Returns: dict with alpha_id, sharpe, fitness, turnover, fail_count, etc.
    """
    if not isinstance(sim_result, dict) or 'id' not in sim_result:
        return None
    
    is_data = sim_result.get('is', {})
    if not is_data:
        return None
    
    checks = is_data.get('checks', [])
    failed = [c.get('name', '') for c in checks if c.get('result') == 'FAIL']
    warnings = [c.get('name', '') for c in checks
                if c.get('result') not in ('PASS', 'FAIL') and c.get('result')]
    
    settings = sim_result.get('settings', {})
    
    return {
        'alpha_id': sim_result['id'],
        'region': settings.get('region', ''),
        'universe': settings.get('universe', ''),
        'sharpe': is_data.get('sharpe', 0),
        'fitness': is_data.get('fitness', 0),
        'turnover': is_data.get('turnover', 0),
        'expression': sim_result.get('regular', {}).get('code', ''),
        'fail_count': len(failed),
        'failed_checks': failed,
        'warnings': warnings,
    }


print("Sanity check functions defined (with extract_sim_record)")

Sanity check functions defined (with extract_sim_record)


## 5. Dataset Parsing Functions

In [123]:
def parse_zero_fail_alphas(filepath: Path) -> List[Dict]:
    """
    Parse dataset file and extract zero-fail alphas
    Returns: List of dicts with keys: block_num, id, sharpe, fitness, turnover, expression
    """
    if not filepath.exists():
        return []

    content = filepath.read_text(encoding="utf-8")
    results = []

    for match in ZERO_FAIL_BLOCK_PATTERN.finditer(content):
        results.append({
            "block_num": int(match.group(1)),
            "id": match.group(2),
            "region": match.group(3),
            "universe": match.group(4),
            "sharpe": float(match.group(5)),
            "fitness": float(match.group(6)),
            "turnover": float(match.group(7)),
            "expression": match.group(8).strip(),
        })

    return results


def load_all_zero_fail_alphas(dataset_files: Dict[str, Path]) -> Dict[str, List[Dict]]:
    """Load zero-fail alphas from all dataset files"""
    all_alphas = {}
    for name, path in dataset_files.items():
        alphas = parse_zero_fail_alphas(path)
        all_alphas[name] = alphas
        print(f"  {name}: {len(alphas)} zero-fail alphas")
    return all_alphas


print("Dataset parsing functions defined")

Dataset parsing functions defined


## 6. Utility Functions

In [124]:
def select_random_datasets(dataset_names: List[str], n: int = 2, rng: random.Random = None) -> List[str]:
    """Randomly select n datasets from the list"""
    if rng is None:
        rng = random.Random()
    if len(dataset_names) < n:
        return []
    return rng.sample(dataset_names, n)


def extract_random_seeds(
    all_alphas: Dict[str, List[Dict]],
    selected_datasets: List[str],
    seeds_per_dataset: int,
    rng: random.Random = None
) -> Tuple[List[Dict], List[str]]:
    """
    Extract random seeds from selected datasets
    Returns: (list of seed dicts with 'dataset' key added, list of dataset names)
    """
    if rng is None:
        rng = random.Random()
    
    seeds = []
    for ds_name in selected_datasets:
        ds_alphas = all_alphas.get(ds_name, [])
        if len(ds_alphas) == 0:
            continue
        
        # Sample up to seeds_per_dataset
        sample_size = min(seeds_per_dataset, len(ds_alphas))
        sampled = rng.sample(ds_alphas, sample_size)
        
        for alpha in sampled:
            alpha_copy = alpha.copy()
            alpha_copy['dataset'] = ds_name
            seeds.append(alpha_copy)
    
    return seeds, selected_datasets


def check_resume_batch(batch_idx: int) -> Tuple[Optional[List[Dict]], bool]:
    """
    Check if batch was already completed (resume support).
    Returns: (combinations_list or None, is_simulated)
    - combinations: list of combinations if JSON exists, None otherwise
    - is_simulated: True if simulation results are already saved
    """
    json_path = GEN_JSON_DIR / f"combinatorial_batch_{batch_idx}.json"
    if json_path.exists():
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            combinations = data.get('combinations', [])
            is_simulated = data.get('simulated', False)
            return combinations, is_simulated
        except:
            return None, False
    return None, False


def save_batch_json(
    batch_idx: int, 
    combinations: List[Dict], 
    selected_datasets: List[str], 
    seeds: List[Dict],
    simulated: bool = False,
    simulation_results: List[Dict] = None
):
    """
    Save batch results to JSON for resume support.
    Includes simulation results if provided.
    """
    GEN_JSON_DIR.mkdir(parents=True, exist_ok=True)
    json_path = GEN_JSON_DIR / f"combinatorial_batch_{batch_idx}.json"
    
    data = {
        'batch_idx': batch_idx,
        'timestamp': datetime.now().isoformat(),
        'datasets': selected_datasets,
        'seed_count': len(seeds),
        'combinations': combinations,
        'simulated': simulated,
    }
    
    if simulation_results is not None:
        data['simulation_results'] = simulation_results
    
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def load_batch_combinations(batch_idx: int) -> Tuple[List[Dict], List[str]]:
    """
    Load combinations from existing batch JSON.
    Returns: (combinations, datasets)
    """
    json_path = GEN_JSON_DIR / f"combinatorial_batch_{batch_idx}.json"
    if json_path.exists():
        try:
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            return data.get('combinations', []), data.get('datasets', [])
        except:
            return [], []
    return [], []


def append_alpha_to_txt(
    filepath: Path,
    alpha_num: int,
    alpha_id: str,
    sharpe: float,
    fitness: float,
    turnover: float,
    expression: str,
    warnings: List[str] = None,
    source_datasets: List[str] = None,
    batch_idx: int = None
):
    """Append a PASS alpha to the output txt file (cumulative)"""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    
    with open(filepath, 'a', encoding='utf-8') as f:
        f.write(f"--- #{alpha_num} | FAIL: 0 (PASS) ---")
        f.write(f"ID: {alpha_id}")
        f.write(f"Region: {REGION}, Universe: {UNIVERSE}")
        f.write(f"Sharpe: {sharpe:.2f}, Fitness: {fitness:.2f}, Turnover: {turnover:.4f}")
        f.write(f"Expression: {expression}")
        if source_datasets:
            f.write(f"Source Datasets: {', '.join(source_datasets)}")
        if batch_idx is not None:
            f.write(f"Batch: {batch_idx}")
        if warnings:
            f.write(f"Warnings: {'; '.join(warnings)}")
        f.write(f"Status: PASS")
        f.write("")


def get_current_alpha_count(filepath: Path) -> int:
    """Count existing alphas in the output file"""
    if not filepath.exists():
        return 0
    content = filepath.read_text(encoding='utf-8')
    return len(re.findall(r'--- #\d+ \| FAIL: 0', content))


print("Utility functions defined (with simulation tracking)")

Utility functions defined (with simulation tracking)


## 7. LLM Prompt Builder

In [ ]:
def format_operators_for_prompt(operators_dict: dict) -> str:
    """Format operators_list.json for LLM prompt with full signatures"""
    ops_by_category = {}
    for op_name, op_info in operators_dict.items():
        cat = op_info.get('category', 'Other')
        if cat not in ops_by_category:
            ops_by_category[cat] = []
        definition = op_info.get('definition', op_name)
        ops_by_category[cat].append(definition)

    lines = []
    for cat, defs in sorted(ops_by_category.items()):
        lines.append(f"[{cat}]")
        for d in defs:
            lines.append(f"  - {d}")
    return "\n".join(lines)


def build_combination_prompt(seeds: List[Dict], operators_dict: dict, num_combinations: int = 100) -> str:
    """
    Build GPT prompt for generating combination alphas from seed expressions.
    """
    seed_list_str = "\n".join([
        f"  {i+1}. [{s['dataset']}] {s['expression']}"
        for i, s in enumerate(seeds)
    ])

    operators_str = format_operators_for_prompt(operators_dict)
    dataset_names = list(set(s['dataset'] for s in seeds))

    prompt = f"""You are an expert in WorldQuant Brain FASTEXPR alpha generation.

<TASK>
Generate {num_combinations} NEW combined alpha expressions by combining the seed alphas below.

RULES:
1. Each combination should use 2-4 seed expressions
2. Freely combine seeds - you choose which ones work well together
3. Use ONLY operators from the ALLOWED_OPERATORS list
4. Follow EXACT operator signatures - wrong argument counts will fail
</TASK>

<SEED_EXPRESSIONS>
You have {len(seeds)} seed expressions from datasets: {', '.join(dataset_names)}

{seed_list_str}
</SEED_EXPRESSIONS>

<ALLOWED_OPERATORS>
CRITICAL: Follow the EXACT signature for each operator.
- If operator shows (x, y) - use EXACTLY 2 arguments
- If operator shows (x) - use EXACTLY 1 argument
- add(x, y) can take 2+ arguments (variadic)
- Never invent operators or use wrong argument counts

{operators_str}
</ALLOWED_OPERATORS>

<RECOMMENDED_IDEA>

When making it, make it with reference to {idea.md}
</RECOMMENDED_IDEA>

<SYNTAX_RULES>
1. Binary operators (EXACTLY 2 args): subtract(a, b), divide(a, b), min(a, b), max(a, b)
2. Unary operators (EXACTLY 1 arg): rank(x), zscore(x), abs(x), log(x), sign(x)
3. Variadic operators (2+ args): add(a, b, ...) can take multiple inputs
4. Time-series (2 args): ts_mean(x, days), ts_std_dev(x, days), ts_delta(x, days)
5. group_zscore(x, group) - EXACTLY 2 args: expression and grouping

COMMON MISTAKES TO AVOID:
- rank(a, b) is WRONG - rank takes only 1 argument
- zscore(a, b) is WRONG - zscore takes only 1 argument
- subtract(a, b, c) is WRONG - subtract takes exactly 2 arguments
</SYNTAX_RULES>

<OUTPUT_FORMAT>
Respond with ONLY a JSON object:

{{{{
  "combinations": [
    {{{{
      "expression": "...",
      "seeds_used": [1, 2],
      "idea": "..."
    }}}},
    ...
  ]
}}}}

IMPORTANT:
- Replace seed references with the FULL seed expression text
- seeds_used: list of seed indices (1-based) that were combined
- Generate exactly {num_combinations} combinations
- Use the min(max(..., -1), 1) clipping pattern
</OUTPUT_FORMAT>

Generate {num_combinations} combinations now."""

    return prompt


def parse_llm_response(response: str, seeds: List[Dict]) -> List[Dict]:
    """Parse LLM response and extract combinations"""
    if not response:
        return []

    json_str = llm.cut_first_to_last_brace(response)
    if not json_str:
        print("  [ERROR] No JSON found in response")
        return []

    try:
        data = json.loads(json_str)
        combinations = data.get('combinations', [])

        valid = []
        for combo in combinations:
            if 'expression' not in combo:
                continue
            expr = combo['expression'].strip()
            if not expr:
                continue

            combo['expression'] = expr
            combo['seeds_used'] = combo.get('seeds_used', [])
            combo['idea'] = combo.get('idea', 'N/A')
            valid.append(combo)

        return valid

    except json.JSONDecodeError as e:
        print(f"  [ERROR] JSON parse failed: {e}")
        return []


print("LLM prompt builder defined (with min/max clipping pattern)")

LLM prompt builder defined (with min/max clipping pattern)


## 8. Main Pipeline Function

In [126]:
def run_pipeline(
    session,
    all_alphas: Dict[str, List[Dict]],
    operators: dict,
    datafields: dict,
    num_batches: int = 10,
    seeds_per_dataset: int = 10,
    combinations_per_batch: int = 100,
    simulation_batch_size: int = 8,
    random_seed: int = 42
):
    """
    Main pipeline: LLM 생성 -> JSON 저장 -> 시뮬레이션 -> 결과 저장
    각 배치에서 가장 좋은 알파를 combined_alphas.txt에 저장
    """
    rng = random.Random(random_seed)
    dataset_names = list(all_alphas.keys())

    total_pass_count = get_current_alpha_count(OUTPUT_TXT_FILE)
    total_simulated = 0
    total_generated = 0

    print("\n" + "="*70)
    print("PIPELINE START")
    print("="*70)
    print(f"Existing PASS alphas: {total_pass_count}")
    print(f"Batches to run: {num_batches}")
    print(f"Available datasets: {dataset_names}")
    print(f"Simulation: 8 alphas at a time")
    print("="*70 + "\n")

    for batch_idx in range(1, num_batches + 1):
        print(f"\n{'='*60}")
        print(f"BATCH {batch_idx}/{num_batches}")
        print(f"{'='*60}")

        # Check resume status
        cached_combinations, is_simulated = check_resume_batch(batch_idx)

        # Case 1: Already simulated -> Skip
        if cached_combinations is not None and is_simulated:
            print(f"[SKIP] Batch {batch_idx} already completed (simulated)")
            continue

        # Case 2: JSON exists but not simulated -> Load and simulate
        if cached_combinations is not None and not is_simulated:
            print(f"[RESUME] Batch {batch_idx} JSON exists, running simulation...")
            valid_combinations = cached_combinations
            _, selected_ds = load_batch_combinations(batch_idx)
            if not selected_ds:
                json_path = GEN_JSON_DIR / f"combinatorial_batch_{batch_idx}.json"
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                selected_ds = data.get('datasets', [])
            seeds = []

        # Case 3: No JSON -> Generate with LLM
        else:
            if len(dataset_names) < 2:
                print("[SKIP] Less than 2 datasets available")
                continue

            selected_ds = select_random_datasets(dataset_names, n=2, rng=rng)
            print(f"\n[1] Selected datasets: {selected_ds}")

            seeds, _ = extract_random_seeds(all_alphas, selected_ds, seeds_per_dataset, rng)
            print(f"[2] Extracted {len(seeds)} seeds ({seeds_per_dataset} per dataset)")

            if len(seeds) < 4:
                print("[SKIP] Not enough seeds extracted")
                continue

            print(f"[3] Calling LLM ({GPT_MODEL}) to generate {combinations_per_batch} combinations...")
            prompt = build_combination_prompt(seeds, operators, combinations_per_batch)

            try:
                response = llm.call_llm_stream(prompt, "", model=GPT_MODEL)
                combinations = parse_llm_response(response, seeds)
                print(f"    LLM returned {len(combinations)} combinations")
            except Exception as e:
                print(f"    [ERROR] LLM call failed: {e}")
                combinations = []

            if not combinations:
                print("[SKIP] No combinations generated")
                save_batch_json(batch_idx, [], selected_ds, seeds, simulated=False)
                continue

            print(f"[4] Sanity checking {len(combinations)} combinations...")
            valid_combinations = []
            for combo in combinations:
                is_valid, error = sanity_checker(combo['expression'], operators, datafields)
                if is_valid:
                    valid_combinations.append(combo)

            print(f"    Valid: {len(valid_combinations)}/{len(combinations)} ({100*len(valid_combinations)/len(combinations):.1f}%)")

            if not valid_combinations:
                print("[SKIP] No valid combinations after sanity check")
                save_batch_json(batch_idx, [], selected_ds, seeds, simulated=True)
                continue

            # Save JSON (not simulated yet)
            save_batch_json(batch_idx, valid_combinations, selected_ds, seeds, simulated=False)
            print(f"    Saved {len(valid_combinations)} combinations to JSON")

        total_generated += len(valid_combinations)

        # ============================================================
        # SIMULATION
        # ============================================================
        alpha_list = []
        tags_list = []
        descs_list = []
        expr_to_combo = {}

        for combo in valid_combinations:
            expr = combo['expression']
            expr_to_combo[expr] = combo

            alpha_list.append({
                "type": "REGULAR",
                "settings": {
                    "instrumentType": "EQUITY",
                    "region": REGION,
                    "universe": UNIVERSE,
                    "delay": DELAY,
                    "decay": 0,
                    "neutralization": "INDUSTRY",
                    "truncation": 0.08,
                    "pasteurization": "ON",
                    "unitHandling": "VERIFY",
                    "nanHandling": "OFF",
                    "language": "FASTEXPR",
                    "visualization": False,
                },
                "regular": expr,
            })

            tags_list.append([f"combine_v2_batch{batch_idx}", f"ds:{'+'.join(selected_ds)}"])
            descs_list.append(f"LLM Combinatorial: {combo.get('idea', 'N/A')[:50]}")

        len_alpha_list = len(alpha_list)
        print(f"[5] Simulating {len_alpha_list} alphas (8 at a time)...")
        batch_pass_count = 0
        simulation_results = []

        # Track best alpha in this batch
        best_alpha_in_batch = None
        best_sharpe_in_batch = -999

        # Simulate in 8-alpha batches
        for rep in range(0, len_alpha_list, simulation_batch_size):

            # Session timeout check
            if ace.check_session_timeout(session) < 500:
                print("    [INFO] Session timeout approaching, refreshing...")
                session = ace.start_session()

            batch_end = min(rep + simulation_batch_size, len_alpha_list)

            try:
                print(f"    --- Simulating {rep+1} to {batch_end} of {len_alpha_list} ---")

                sim_results = list(
                    ace.multi_simulate_alphas_map(
                        session,
                        alpha_list[rep:batch_end],
                        tags_list[rep:batch_end],
                        descs_list[rep:batch_end],
                        batch_end - rep
                    )
                )

                # Process each result
                for result in sim_results:
                    record = extract_sim_record(result)
                    if record is None:
                        continue

                    total_simulated += 1
                    simulation_results.append(record)

                    alpha_id = record['alpha_id']
                    sharpe = record['sharpe']
                    fitness = record['fitness']
                    turnover = record['turnover']
                    fail_count = record['fail_count']
                    failed_checks = record['failed_checks']
                    warnings = record['warnings']

                    original_expr = record.get('expression', '')
                    combo = expr_to_combo.get(original_expr)
                    if combo is None:
                        for expr, c in expr_to_combo.items():
                            if expr[:50] == original_expr[:50]:
                                combo = c
                                break

                    if combo is None:
                        print(f"      {alpha_id}: Sharpe={sharpe:.2f}, Fitness={fitness:.2f}, FAIL={fail_count} (combo not matched)")
                        continue

                    # Track best alpha in batch (by Sharpe, regardless of fail_count)
                    if sharpe > best_sharpe_in_batch:
                        best_sharpe_in_batch = sharpe
                        best_alpha_in_batch = {
                            'alpha_id': alpha_id,
                            'sharpe': sharpe,
                            'fitness': fitness,
                            'turnover': turnover,
                            'expression': combo['expression'],
                            'fail_count': fail_count,
                            'warnings': warnings
                        }

                    if fail_count == 0:
                        total_pass_count += 1
                        batch_pass_count += 1
                        print(f"      v {alpha_id}: Sharpe={sharpe:.2f}, Fitness={fitness:.2f}, PASS (#{total_pass_count})")
                    else:
                        fail_str = ', '.join(failed_checks[:3])
                        if len(failed_checks) > 3:
                            fail_str += f" +{len(failed_checks)-3}"
                        print(f"      x {alpha_id}: Sharpe={sharpe:.2f}, Fitness={fitness:.2f}, FAIL={fail_count} [{fail_str}]")

            except Exception as e:
                print(f"    [ERROR] Simulation failed for batch {rep}-{batch_end}: {e}")
                continue

        # Save best alpha from this batch to combined_alphas.txt
        if best_alpha_in_batch is not None:
            total_pass_count += 1
            append_alpha_to_txt(
                OUTPUT_TXT_FILE,
                alpha_num=total_pass_count,
                alpha_id=best_alpha_in_batch['alpha_id'],
                sharpe=best_alpha_in_batch['sharpe'],
                fitness=best_alpha_in_batch['fitness'],
                turnover=best_alpha_in_batch['turnover'],
                expression=best_alpha_in_batch['expression'],
                warnings=best_alpha_in_batch['warnings'] if best_alpha_in_batch['warnings'] else None,
                source_datasets=selected_ds,
                batch_idx=batch_idx
            )
            print(f"\n    [BEST] Saved best alpha: Sharpe={best_alpha_in_batch['sharpe']:.2f}, ID={best_alpha_in_batch['alpha_id']}")

        # Update JSON with simulation results (mark as simulated)
        save_batch_json(
            batch_idx,
            valid_combinations,
            selected_ds,
            seeds,
            simulated=True,
            simulation_results=simulation_results
        )
        print(f"    Batch {batch_idx} complete: {batch_pass_count} PASS, best Sharpe={best_sharpe_in_batch:.2f}")

    print("\n" + "="*70)
    print("PIPELINE COMPLETE")
    print("="*70)
    print(f"Total batches run: {num_batches}")
    print(f"Total combinations generated: {total_generated}")
    print(f"Total simulated: {total_simulated}")
    print(f"Total saved alphas: {total_pass_count}")
    print(f"Output file: {OUTPUT_TXT_FILE}")
    print("="*70)

    return session


print("Main pipeline function defined (saves best alpha per batch)")

Main pipeline function defined (saves best alpha per batch)


## 9. Load Resources

In [127]:
print("Loading resources...\n")

# Load zero-fail alphas from all datasets
print("Zero-fail alphas:")
all_alphas = load_all_zero_fail_alphas(DATASET_FILES)

# Load operators
operators = llm.import_json(str(OPERATORS_FILE))
print(f"\nOperators: {len(operators)}")

# Load datafields
datafields = llm.import_json(str(DATAFIELDS_FILE))
print(f"Datafields: {len(datafields)}")

print("\nResources loaded!")

Loading resources...

Zero-fail alphas:
  model25: 52 zero-fail alphas
  model30: 29 zero-fail alphas
  model138: 151 zero-fail alphas
  analyst39: 12 zero-fail alphas

Operators: 83
Datafields: 1932

Resources loaded!


## 10. Brain Session Login

In [128]:
print("Starting Brain API session...")
session = ace.start_session()
print("Session established!")

Starting Brain API session...
Complete biometrics authentication and press any key to continue: 
https://api.worldquantbrain.com/authentication/persona?inquiry=inq_fkuZhSnqTQaUpcqpgf5BqPde4ywi

Session established!


## 11. Run Pipeline

In [129]:
run_pipeline(
    session=session,
    all_alphas=all_alphas,
    operators=operators,
    datafields=datafields,
    num_batches=NUM_BATCHES,
    seeds_per_dataset=SEEDS_PER_DATASET,
    combinations_per_batch=COMBINATIONS_PER_BATCH,
    simulation_batch_size=SIMULATION_BATCH_SIZE,
    random_seed=RANDOM_SEED
)


PIPELINE START
Existing PASS alphas: 37
Batches to run: 100
Available datasets: ['model25', 'model30', 'model138', 'analyst39']
Simulation: 8 alphas at a time


BATCH 1/100

[1] Selected datasets: ['model25', 'analyst39']
[2] Extracted 20 seeds (10 per dataset)
[3] Calling LLM (gpt-4o) to generate 100 combinations...
```json
{
  "combinations": [
    {
      "expression": "min(max(add(multiply(ts_zscore(mdl25_vrv421_91v, 42), rank(mdl25_vrv421_91v)), ts_zscore(rank(mdl25_vrv421_71v), 126)), -1), 1)",
      "seeds_used": [1, 2],
      "idea": "Combine two different z-scores with a rank to create a robust alpha by adding multiplying components."
    },
    {
      "expression": "min(max(multiply(ts_zscore(mdl25_vrv421_71v, 100), rank(ts_rank(anl39_qtanbvps, 126))), -1), 1)",
      "seeds_used": [3, 11],
      "idea": "Using time series z-scores of model25 data with the ranked output of analyst39 cross-sectionally."
    },
    {
      "expression": "min(max(add(winsorize(ts_zscore(mdl25_

2026-02-09 21:12:52,481 - ace - WARNING - Simulation failed. {"detail":"CONCURRENT_SIMULATION_LIMIT_EXCEEDED"}, Status code: 429


{'id': 'A1J7d5Al', 'type': 'REGULAR', 'author': 'PS40111', 'settings': {'instrumentType': 'EQUITY', 'region': 'EUR', 'universe': 'TOP2500', 'delay': 1, 'decay': 0, 'neutralization': 'INDUSTRY', 'truncation': 0.08, 'pasteurization': 'ON', 'unitHandling': 'VERIFY', 'nanHandling': 'OFF', 'maxTrade': 'OFF', 'language': 'FASTEXPR', 'visualization': False, 'startDate': '2014-01-01', 'endDate': '2023-12-31'}, 'regular': {'code': 'min(max(multiply(ts_zscore(mdl25_vrv421_91v, 252), ts_zscore(anl39_ttmepsincx, 252)), -1), 1)', 'description': None, 'operatorCount': 5}, 'dateCreated': '2026-02-09T07:14:09-05:00', 'dateSubmitted': None, 'dateModified': '2026-02-09T07:14:09-05:00', 'name': None, 'favorite': False, 'hidden': False, 'color': None, 'category': None, 'tags': ['ace_tag', 'combine_v2_batch1', 'ds:model25+analyst39'], 'classifications': [], 'grade': None, 'stage': 'IS', 'status': 'UNSUBMITTED', 'is': {'pnl': 6358, 'bookSize': 20000000, 'longCount': 1169, 'shortCount': 1647, 'turnover': 0.0

2026-02-09 21:17:06,654 - ace - WARNING - Simulation failed. {"detail":"CONCURRENT_SIMULATION_LIMIT_EXCEEDED"}, Status code: 429


{'id': 'Vkw3nEEM', 'type': 'REGULAR', 'author': 'PS40111', 'settings': {'instrumentType': 'EQUITY', 'region': 'EUR', 'universe': 'TOP2500', 'delay': 1, 'decay': 0, 'neutralization': 'INDUSTRY', 'truncation': 0.08, 'pasteurization': 'ON', 'unitHandling': 'VERIFY', 'nanHandling': 'OFF', 'maxTrade': 'OFF', 'language': 'FASTEXPR', 'visualization': False, 'startDate': '2014-01-01', 'endDate': '2023-12-31'}, 'regular': {'code': 'min(max(add(winsorize(ts_zscore(mdl25_vrv421_91v, 63), std=4), ts_zscore(anl39_qtanbvps, 14)), -1), 1)', 'description': None, 'operatorCount': 6}, 'dateCreated': '2026-02-09T07:18:16-05:00', 'dateSubmitted': None, 'dateModified': '2026-02-09T07:18:17-05:00', 'name': None, 'favorite': False, 'hidden': False, 'color': None, 'category': None, 'tags': ['ace_tag', 'combine_v2_batch1', 'ds:model25+analyst39'], 'classifications': [], 'grade': None, 'stage': 'IS', 'status': 'UNSUBMITTED', 'is': {'pnl': 1376602, 'bookSize': 20000000, 'longCount': 297, 'shortCount': 2367, 'tur

...

{'id': 'ak2EXl62', 'type': 'REGULAR', 'author': 'PS40111', 'settings': {'instrumentType': 'EQUITY', 'region': 'EUR', 'universe': 'TOP2500', 'delay': 1, 'decay': 0, 'neutralization': 'INDUSTRY', 'truncation': 0.08, 'pasteurization': 'ON', 'unitHandling': 'VERIFY', 'nanHandling': 'OFF', 'maxTrade': 'OFF', 'language': 'FASTEXPR', 'visualization': False, 'startDate': '2014-01-01', 'endDate': '2023-12-31'}, 'regular': {'code': 'min(max(multiply(sqrt(ts_zscore(mdl25_vrv421_71v, 126)), sqrt(ts_zscore(anl39_ttmepsincx, 252))), -1), 1)', 'description': None, 'operatorCount': 7}, 'dateCreated': '2026-02-09T07:28:28-05:00', 'dateSubmitted': None, 'dateModified': '2026-02-09T07:28:28-05:00', 'name': None, 'favorite': False, 'hidden': False, 'color': None, 'category': None, 'tags': ['ace_tag', 'combine_v2_batch1', 'ds:model25+analyst39'], 'classifications': [], 'grade': None, 'stage': 'IS', 'status': 'UNSUBMITTED', 'is': {'pnl': 3866048, 'bookSize': 20000000, 'longCount': 581, 'shortCount': 2197, '

2026-02-09 21:49:15,540 - ace - ERROR - Simulation failed. {'id': '1DdBsy1J94ggcuX4AdpJexW', 'type': 'REGULAR', 'status': 'ERROR', 'message': 'Unknown attribute "filter" encountered', 'location': {'line': 1, 'start': 8, 'end': 105, 'property': 'regular'}}
2026-02-09 21:49:16,140 - ace - ERROR - Simulation failed. {'id': '2CI9YRdvH4juanYN1VCCRwQ', 'type': 'REGULAR', 'status': 'ERROR', 'message': 'Unexpected character \',\' near "00))), -1), 1)"', 'location': {'line': 1, 'start': 130, 'end': 131, 'property': 'regular'}}


{'id': 'MPlLqxO8', 'type': 'REGULAR', 'author': 'PS40111', 'settings': {'instrumentType': 'EQUITY', 'region': 'EUR', 'universe': 'TOP2500', 'delay': 1, 'decay': 0, 'neutralization': 'INDUSTRY', 'truncation': 0.08, 'pasteurization': 'ON', 'unitHandling': 'VERIFY', 'nanHandling': 'OFF', 'maxTrade': 'OFF', 'language': 'FASTEXPR', 'visualization': False, 'startDate': '2014-01-01', 'endDate': '2023-12-31'}, 'regular': {'code': 'min(max(if_else(ts_zscore(rank(vec_avg(mdl138_3idpc)), 252) < ts_zscore(anl39_roxlcxspeq, 252), rank(vec_avg(mdl138_5idpc)), subtract(ts_zscore(anl39_roxlcxspeq, 252), ts_backfill(vec_avg(mdl138_4idpqc), 126))), -1), 1)', 'description': None, 'operatorCount': 14}, 'dateCreated': '2026-02-09T07:50:23-05:00', 'dateSubmitted': None, 'dateModified': '2026-02-09T07:50:24-05:00', 'name': None, 'favorite': False, 'hidden': False, 'color': None, 'category': None, 'tags': ['ace_tag', 'combine_v2_batch6', 'ds:analyst39+model138'], 'classifications': [], 'grade': None, 'stage':

KeyboardInterrupt: 

{'id': 'KPYEqodN', 'type': 'REGULAR', 'author': 'PS40111', 'settings': {'instrumentType': 'EQUITY', 'region': 'EUR', 'universe': 'TOP2500', 'delay': 1, 'decay': 0, 'neutralization': 'INDUSTRY', 'truncation': 0.08, 'pasteurization': 'ON', 'unitHandling': 'VERIFY', 'nanHandling': 'OFF', 'maxTrade': 'OFF', 'language': 'FASTEXPR', 'visualization': False, 'startDate': '2014-01-01', 'endDate': '2023-12-31'}, 'regular': {'code': 'min(max(add(min(ts_mean(mdl25_vrv421_71v, 21), ts_zscore(mdl25_vrv421_71v, 63)), ts_zscore(anl39_ttmepsincx, 252)), -1), 1)', 'description': None, 'operatorCount': 7}, 'dateCreated': '2026-02-09T07:55:33-05:00', 'dateSubmitted': None, 'dateModified': '2026-02-09T07:55:34-05:00', 'name': None, 'favorite': False, 'hidden': False, 'color': None, 'category': None, 'tags': ['ace_tag', 'combine_v2_batch7', 'ds:model25+analyst39'], 'classifications': [], 'grade': None, 'stage': 'IS', 'status': 'UNSUBMITTED', 'is': {'pnl': 3480238, 'bookSize': 20000000, 'longCount': 1029, 's

## 12. View Results

In [ ]:
if OUTPUT_TXT_FILE.exists():
    content = OUTPUT_TXT_FILE.read_text(encoding='utf-8')
    lines = content.split('\n')
    
    print(f"Output file: {OUTPUT_TXT_FILE}")
    print(f"Total lines: {len(lines)}")
    print(f"PASS alphas: {get_current_alpha_count(OUTPUT_TXT_FILE)}")
    print("\n" + "="*60)
    print("First 50 lines:")
    print("="*60)
    print('\n'.join(lines[:50]))
else:
    print(f"No output file yet: {OUTPUT_TXT_FILE}")

Output file: C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\results\combinatorial\combined_alphas.txt
Total lines: 1
PASS alphas: 37

First 50 lines:
--- #1 | FAIL: 0 (PASS) ---ID: LLb1zqv1Region: EUR, Universe: TOP2500Sharpe: 2.27, Fitness: 1.51, Turnover: 0.1235Expression: add(rank(ts_backfill(mdl30_psprise_pct_fy1_eps, 10)), ts_zscore(rank(vec_avg(mdl138_5idpqc)), 126))Source Datasets: model30, model138Batch: 2Warnings: CONCENTRATED_WEIGHT; SELF_CORRELATION; DATA_DIVERSITY; PROD_CORRELATION; REGULAR_SUBMISSION; IS_LADDER_SHARPE; POWER_POOL_CORRELATION; MATCHES_COMPETITION; MATCHES_THEMESStatus: PASS--- #2 | FAIL: 0 (PASS) ---ID: mL2b7Q0pRegion: EUR, Universe: TOP2500Sharpe: 2.07, Fitness: 0.85, Turnover: 0.2782Expression: add(rank(mdl30_psprise_pct_fy1_eps), rank(star_eps_surprise_prediction_fy1), ts_zscore(mdl30_new_psprise_pct_f12m_eps, 30))Source Datasets: analyst39, model30Batch: 5Warnings: LOW_FITNESS; CONCENTRATED_WEIGHT; SELF_CORRELATION; DATA_DIVERSITY; PROD_CORRELATION; REGULAR